In [0]:
%run /Workspace/Users/molugurikoushik68@gmail.com/banking-agent/notebooks/02_embeddings_generator

In [0]:
"""
Vector Search & Storage
=======================

Stores embeddings in Databricks table and implements semantic search.

Process:
1. Load embeddings from previous step
2. Create Databricks table for storage
3. Insert embeddings into table
4. Implement search function
5. Test with sample queries
"""

import pandas as pd
import numpy as np
from typing import List, Tuple, Dict
import math

# ============================================
# 1. FUNCTION: Create Databricks Table
# ============================================

def create_embeddings_table(schema: str, table_name: str):
    """
    Create Databricks table to store embeddings.
    
    What it does:
    1. Drops table if exists (clean start)
    2. Creates new table with columns:
       - chunk_id: unique identifier
       - text: policy text
       - embedding: vector (1024 numbers)
       - word_count: how many words in chunk
       - char_count: how many characters
    3. Sets up for efficient search
    
    Why:
    - Delta table is Databricks' optimized storage
    - Can store complex data types (vectors)
    - Supports transactions and updates
    
    Args:
        schema: Database name (e.g., "banking_agent_db")
        table_name: Table name (e.g., "policy_embeddings")
    """
    
    table_path = f"{schema}.{table_name}"
    
    print(f"Creating table: {table_path}")
    
    try:
        # Drop existing table
        spark.sql(f"DROP TABLE IF EXISTS {table_path}")
        print(f"  ✓ Dropped existing table")
    except:
        pass
    
    try:
        # Create table using SQL
        create_table_sql = f"""
        CREATE TABLE {table_path} (
            chunk_id INT,
            text STRING,
            embedding ARRAY<DOUBLE>,
            word_count INT,
            char_count INT
        )
        USING DELTA
        """
        
        spark.sql(create_table_sql)
        print(f"  ✓ Created new table")
        print(f"  ✓ Columns: chunk_id, text, embedding, word_count, char_count")
        
    except Exception as e:
        print(f"  ✗ Error creating table: {e}")

# ============================================
# 2. FUNCTION: Insert Embeddings into Table
# ============================================

def insert_embeddings(embeddings_df: pd.DataFrame, 
                     schema: str, 
                     table_name: str):
    """
    Insert embeddings dataframe into Databricks table.
    
    What it does:
    1. Take Pandas dataframe with embeddings
    2. Convert to Spark dataframe
    3. Insert into Databricks table
    4. Verify insertion
    
    Why:
    - Stores embeddings persistently
    - Makes them queryable
    - Enables semantic search
    
    Args:
        embeddings_df: Pandas dataframe with embeddings
        schema: Database name
        table_name: Table name
    """
    
    table_path = f"{schema}.{table_name}"
    
    # Convert data types to match Delta table schema
    embeddings_dataframe_fixed = embeddings_df.copy()
    embeddings_dataframe_fixed['chunk_id'] = embeddings_dataframe_fixed['chunk_id'].astype('int32')
    embeddings_dataframe_fixed['word_count'] = embeddings_dataframe_fixed['word_count'].astype('int32')
    embeddings_dataframe_fixed['char_count'] = embeddings_dataframe_fixed['char_count'].astype('int32')
    
    # Insert embeddings
    print(f"Inserting {len(embeddings_dataframe_fixed)} embeddings...")
    
    try:
        spark_df = spark.createDataFrame(embeddings_dataframe_fixed)
        spark_df = spark_df.select('chunk_id', 'text', 'embedding', 'word_count', 'char_count')
        spark_df.write.mode("append").format("delta").saveAsTable(table_path)
        
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_path}").collect()[0]['cnt']
        print(f"  ✓ Inserted {count} embeddings")
        
    except Exception as e:
        print(f"  ✗ Error inserting embeddings: {e}")
    
    

# ============================================
# 3. FUNCTION: Cosine Similarity (Python)
# ============================================

def cosine_similarity_py(vec1: List[float], vec2: List[float]) -> float:
    """
    Calculate cosine similarity between two vectors (Python version).
    
    Args:
        vec1: First vector
        vec2: Second vector
        
    Returns:
        Similarity score (0.0 to 1.0)
    """
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    mag1 = math.sqrt(sum(x**2 for x in vec1))
    mag2 = math.sqrt(sum(x**2 for x in vec2))
    
    if mag1 == 0 or mag2 == 0:
        return 0.0
    
    return dot_product / (mag1 * mag2)

# ============================================
# 4. FUNCTION: Search Function (SQL)
# ============================================

def search_policies_sql(query_embedding: List[float],
                       schema: str,
                       table_name: str,
                       top_k: int = 3,
                       min_similarity: float = 0.5) -> List[Dict]:
    """
    Search for relevant policies using SQL.
    
    What it does:
    1. Calculate similarity between query and all embeddings
    2. Filter by minimum similarity threshold
    3. Return top K results sorted by similarity
    
    Why SQL?
    - Fast (Databricks optimizes it)
    - Can handle large tables
    - Part of Databricks ecosystem
    
    Args:
        query_embedding: Embedding of query
        schema: Database name
        table_name: Table name
        top_k: Return top K results
        min_similarity: Minimum similarity threshold
        
    Returns:
        List of similar policies with scores
    """
    
    table_path = f"{schema}.{table_name}"
    
    # Convert query to format for SQL
    query_str = json.dumps(query_embedding)
    
    # SQL query to calculate similarities
    # Note: Databricks Delta doesn't have built-in vector similarity,
    # so we use Python UDF or calculate in Spark
    
    try:
        sql = f"""
        SELECT 
            chunk_id,
            text,
            word_count,
            embedding
        FROM {table_path}
        ORDER BY chunk_id
        LIMIT {top_k * 2}
        """
        
        results_spark = spark.sql(sql)
        results = results_spark.collect()
        
        # Calculate similarities in Python
        similarities = []
        for row in results:
            similarity = cosine_similarity_py(query_embedding, list(row['embedding']))
            if similarity >= min_similarity:
                similarities.append({
                    'chunk_id': row['chunk_id'],
                    'text': row['text'],
                    'word_count': row['word_count'],
                    'similarity': similarity
                })
        
        # Sort by similarity (descending)
        similarities.sort(key=lambda x: x['similarity'], reverse=True)
        
        return similarities[:top_k]
        
    except Exception as e:
        print(f"✗ Error searching: {e}")
        return []

# ============================================
# 5. FUNCTION: Search Policies (Main)
# ============================================

def search_policies(query_text: str,
                   query_embedding: List[float],
                   schema: str,
                   table_name: str,
                   top_k: int = 3,
                   min_similarity: float = 0.5) -> List[Dict]:
    """
    Main search function: Find policies relevant to query.
    
    What it does:
    1. Take a query (e.g., "Can I transfer $5000?")
    2. Use embedding to find similar policies
    3. Return top K most relevant policies
    
    Why:
    - This is what the agent uses to find context
    - Semantic search (not just keyword matching)
    - Fast retrieval
    
    Args:
        query_text: User query (e.g., "Can I transfer $5000?")
        query_embedding: Embedding of query
        schema: Database name
        table_name: Table name
        top_k: Return top K results (default 3)
        min_similarity: Minimum similarity (default 0.5)
        
    Returns:
        List of relevant policies
        
    Example:
        results = search_policies(
            "What is transfer limit?",
            query_embedding=[...],
            schema="banking_agent_db",
            table_name="policy_embeddings",
            top_k=3
        )
        # Returns top 3 policies about transfers
    """
    
    print(f"Searching for: '{query_text}'")
    print(f"Parameters: top_k={top_k}, min_similarity={min_similarity}")
    
    # Search using SQL
    results = search_policies_sql(
        query_embedding,
        schema,
        table_name,
        top_k=top_k,
        min_similarity=min_similarity
    )
    
    print(f"Found {len(results)} relevant policies:\n")
    
    for rank, result in enumerate(results, 1):
        print(f"{rank}. Chunk {result['chunk_id']} (similarity: {result['similarity']:.4f})")
        print(f"   {result['text'][:100]}...")
        print()
    
    return results

# ============================================
# 6. FUNCTION: Display Search Statistics
# ============================================

def display_search_stats(schema: str, table_name: str):
    """
    Display statistics about stored embeddings.
    """
    
    table_path = f"{schema}.{table_name}"
    
    print(f"\nTable Statistics: {table_path}")
    print("="*60)
    
    try:
        # Get table info
        stats = spark.sql(f"""
        SELECT 
            COUNT(*) as total_embeddings,
            SUM(word_count) as total_words,
            AVG(word_count) as avg_words,
            MIN(word_count) as min_words,
            MAX(word_count) as max_words
        FROM {table_path}
        """).collect()[0]
        
        print(f"✓ Total embeddings: {stats['total_embeddings']}")
        print(f"✓ Total words: {stats['total_words']}")
        print(f"✓ Avg words per chunk: {stats['avg_words']:.0f}")
        print(f"✓ Min/Max words: {stats['min_words']}/{stats['max_words']}")
        
    except Exception as e:
        print(f"✗ Error getting stats: {e}")

# ============================================
# 7. MAIN EXECUTION
# ============================================

import json

# Configuration and embeddings are already loaded from Cell 1 (%run 02_embeddings_generator)
# Available: CATALOG, SCHEMA, EMBEDDINGS_TABLE, TOP_K, MIN_SIMILARITY
# Available: embeddings_for_storage, embeddings_dataframe

print("="*60)
print("VECTOR SEARCH: Storing & Retrieving Embeddings")
print("="*60)

# Step 1: Create table
print("\nStep 1: Creating Databricks table...")
create_embeddings_table(SCHEMA, EMBEDDINGS_TABLE)

# Step 2: Insert embeddings
print("\nStep 2: Inserting embeddings...")
insert_embeddings(embeddings_dataframe, SCHEMA, EMBEDDINGS_TABLE)

# Step 3: Display statistics
print("\nStep 3: Table statistics...")
display_search_stats(SCHEMA, EMBEDDINGS_TABLE)

# Step 4: Test searches
print("\n" + "="*60)
print("TESTING SEMANTIC SEARCH")
print("="*60)

# Get the embedding function to embed the queries
from openai import OpenAI

client = OpenAI(
    base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints",
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

def get_query_embedding(text: str) -> List[float]:
    response = client.embeddings.create(
        model="databricks-bge-large-en",
        input=text
    )
    return response.data[0].embedding

# Test with REAL query embeddings
test_queries = [
    {
        'text': "What is the transfer limit?",
        'embedding': get_query_embedding("What is the transfer limit?")
    },
    {
        'text': "How long does a refund take?",
        'embedding': get_query_embedding("How long does a refund take?")
    },
    {
        'text': "What is minimum account balance?",
        'embedding': get_query_embedding("What is minimum account balance?")
    }
]

print(f"\nTesting {len(test_queries)} queries...\n")

for query_data in test_queries:
    print("-" * 60)
    search_results = search_policies(
        query_data['text'],
        query_data['embedding'],
        SCHEMA,
        EMBEDDINGS_TABLE,
        top_k=TOP_K,
        min_similarity=MIN_SIMILARITY
    )
    print()

# Step 5: Export search function for agent
print("="*60)
print("SUMMARY")
print("="*60)
print(f"✓ Created table: {SCHEMA}.{EMBEDDINGS_TABLE}")
print(f"✓ Stored {len(embeddings_dataframe)} embeddings")
print(f"✓ Implemented semantic search")
print(f"✓ Ready for agent integration (Phase 3)")

# Export search function
search_function = search_policies

print(f"\n✓ Search function exported as 'search_function'")
print(f"✓ Usage: search_function(query_text, query_embedding, schema, table_name)")